Notebook 4: Zero-Shot LLM Evaluation
================================================================================

Overview
--------
This notebook evaluates a large language model (LLM) using zero-shot prompting
for greenwashing detection WITHOUT any fine-tuning.

Hypothesis
----------
Modern large language models like Llama-3 or Mistral can perform classification
tasks through prompting alone, without requiring training data. We test whether
zero-shot LLM performance can compete with fine-tuned smaller models.

Process
-------
1. Load evaluation data (from Notebook 1)
2. Load previous model results (RoBERTa, FinBERT)
3. Set up LLM (Ollama or Transformers API)
4. Design classification prompt
5. Evaluate on test set
6. Compare all three models

Requirements Met
----------------
- Add Model 3 (Zero-Shot LLM)
- Compare three different model architectures
- Provide final comparative analysis

## Setup & Imports

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

print("Libraries loaded successfully.")

# Check for available LLM backends
try:
    from transformers import pipeline
    HF_AVAILABLE = True
    print("✓ Hugging Face Transformers available")
except ImportError:
    HF_AVAILABLE = False
    print("✗ Hugging Face Transformers not available")

try:
    import requests
    OLLAMA_AVAILABLE = True
    print("✓ Requests available (for Ollama)")
except ImportError:
    OLLAMA_AVAILABLE = False
    print("✗ Requests not available")


## Configuration

In [ ]:
# Data paths
EVAL_PATH = "../inputs/eval_synthetic.csv"
RESULTS_DIR = "../outputs"

# LLM Configuration
# Choose your backend: "ollama" or "transformers"
LLM_BACKEND = "transformers"  # Change to "ollama" if you have Ollama installed

# Model selection
if LLM_BACKEND == "transformers":
    MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # Smaller model for CPU
    # Alternatives: "mistralai/Mistral-7B-Instruct-v0.1", "microsoft/phi-2"
elif LLM_BACKEND == "ollama":
    MODEL_NAME = "llama3:8b"  # Requires Ollama to be running
    OLLAMA_URL = "http://localhost:11434/api/generate"

# Evaluation settings
MAX_SAMPLES = None  # Set to a number (e.g., 50) for quick testing
BATCH_SIZE = 1  # Process one at a time for stability

# Create output directory
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"\nLLM Backend: {LLM_BACKEND}")
print(f"Model: {MODEL_NAME}")

## Load Evaluation Data

In [ ]:
print("\n" + "="*60)
print("LOADING EVALUATION DATA")
print("="*60)

# Load test set
eval_df = pd.read_csv(EVAL_PATH)

# Limit samples for testing if specified
if MAX_SAMPLES:
    eval_df = eval_df.sample(MAX_SAMPLES, random_state=42)
    print(f"Using {MAX_SAMPLES} samples for quick testing")

print(f"Evaluation set: {len(eval_df)} samples")
print(f"\nClass distribution:")
print(eval_df['label'].value_counts())

## Load Previous Model Results

In [ ]:
print("\n" + "="*60)
print("LOADING PREVIOUS MODEL RESULTS")
print("="*60)

# Load RoBERTa results
roberta_path = os.path.join(RESULTS_DIR, "roberta_baseline_results.json")
if os.path.exists(roberta_path):
    with open(roberta_path, 'r') as f:
        roberta_results = json.load(f)
    print(f"✓ RoBERTa: {roberta_results['finetuned_accuracy']:.4f} accuracy")
else:
    roberta_results = {"finetuned_accuracy": 0.95}
    print("⚠️  RoBERTa results not found, using placeholder")

# Load FinBERT results
finbert_path = os.path.join(RESULTS_DIR, "finbert_results.json")
if os.path.exists(finbert_path):
    with open(finbert_path, 'r') as f:
        finbert_results = json.load(f)
    print(f"✓ FinBERT: {finbert_results['finetuned_accuracy']:.4f} accuracy")
else:
    finbert_results = {"finetuned_accuracy": 0.96}
    print("⚠️  FinBERT results not found, using placeholder")

## Define Classification Prompt

In [ ]:
print("\n" + "="*60)
print("PROMPT DESIGN")
print("="*60)

SYSTEM_PROMPT = """You are an expert in analyzing corporate sustainability reports. 
Your task is to classify statements as either SPECIFIC or VAGUE.

SPECIFIC statements contain:
- Concrete numbers (percentages, amounts)
- Specific dates or deadlines (e.g., "by 2030")
- Measurable units (tons CO2, GWh, EUR)
- Clear, verifiable commitments

VAGUE statements contain:
- Hedging words (may, might, aim, intend, seek)
- General aspirations without concrete targets
- No specific numbers or dates
- Ambiguous commitments

Respond with ONLY one word: either "SPECIFIC" or "VAGUE"."""

def create_prompt(sentence):
    """Create the full prompt for classification."""
    return f"""{SYSTEM_PROMPT}

Classify this sentence:
"{sentence}"

Classification:"""

print("Prompt template created.")
print("\nExample prompt:")
print("-" * 60)
example = create_prompt("We reduced emissions by 42% compared to 2020.")
print(example[:300] + "...")

## Setup LLM Backend

In [ ]:
print("\n" + "="*60)
print("SETTING UP LLM BACKEND")
print("="*60)

if LLM_BACKEND == "transformers":
    print(f"Loading model: {MODEL_NAME}")
    print("This may take several minutes on first run...")
    
    try:
        llm_pipeline = pipeline(
            "text-generation",
            model=MODEL_NAME,
            max_new_tokens=10,
            do_sample=False,
            device=-1  # CPU, change to 0 for GPU
        )
        print("✓ Model loaded successfully")
    except Exception as e:
        print(f"✗ Error loading model: {e}")
        print("\nTrying smaller alternative model: microsoft/phi-2")
        MODEL_NAME = "microsoft/phi-2"
        llm_pipeline = pipeline(
            "text-generation",
            model=MODEL_NAME,
            max_new_tokens=10,
            do_sample=False,
            device=-1
        )

elif LLM_BACKEND == "ollama":
    print(f"Using Ollama model: {MODEL_NAME}")
    print(f"Ollama URL: {OLLAMA_URL}")
    
    # Test Ollama connection
    try:
        response = requests.post(
            OLLAMA_URL,
            json={"model": MODEL_NAME, "prompt": "test", "stream": False}
        )
        if response.status_code == 200:
            print("✓ Ollama connection successful")
        else:
            print(f"✗ Ollama error: {response.status_code}")
    except Exception as e:
        print(f"✗ Cannot connect to Ollama: {e}")
        print("\nMake sure Ollama is running: ollama serve")

## Classification Functions

In [ ]:
def classify_with_transformers(sentence):
    """Classify using Hugging Face Transformers."""
    prompt = create_prompt(sentence)
    try:
        response = llm_pipeline(prompt)[0]['generated_text']
        # Extract classification from response
        response = response.split("Classification:")[-1].strip().upper()
        
        if "SPECIFIC" in response[:20]:
            return "SPECIFIC"
        elif "VAGUE" in response[:20]:
            return "VAGUE"
        else:
            return "VAGUE"  # Default to VAGUE if unclear
    except Exception as e:
        print(f"Error in classification: {e}")
        return "VAGUE"

def classify_with_ollama(sentence):
    """Classify using Ollama."""
    prompt = create_prompt(sentence)
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False
            },
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()['response'].strip().upper()
            
            if "SPECIFIC" in result[:20]:
                return "SPECIFIC"
            elif "VAGUE" in result[:20]:
                return "VAGUE"
            else:
                return "VAGUE"
        else:
            return "VAGUE"
    except Exception as e:
        print(f"Error: {e}")
        return "VAGUE"

# Select classification function based on backend
if LLM_BACKEND == "transformers":
    classify_function = classify_with_transformers
else:
    classify_function = classify_with_ollama

print("✓ Classification function ready")

## Zero-Shot Evaluation
##
### Run the LLM on each test sample without any training.

In [ ]:
print("\n" + "="*60)
print("ZERO-SHOT EVALUATION")
print("="*60)
print(f"Evaluating {len(eval_df)} samples...")
print("This may take 10-30 minutes depending on model and hardware.\n")

predictions = []
true_labels = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Classifying"):
    sentence = row['text']
    true_label = row['label']
    
    # Get prediction
    pred = classify_function(sentence)
    
    # Convert to numeric
    pred_label = 1 if pred == "SPECIFIC" else 0
    
    predictions.append(pred_label)
    true_labels.append(true_label)
    
    # Small delay to avoid rate limiting
    time.sleep(0.1)

predictions = np.array(predictions)
true_labels = np.array(true_labels)

print("\n✓ Evaluation complete")

## Calculate Metrics

In [ ]:
print("\n" + "="*60)
print("ZERO-SHOT LLM RESULTS")
print("="*60)

from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support, 
    classification_report,
    confusion_matrix
)

# Calculate metrics
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, predictions, average='weighted'
)

print(f"\nZero-Shot LLM ({MODEL_NAME}):")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    true_labels,
    predictions,
    target_names=["Vague", "Specific"],
    digits=4
))

print("\nConfusion Matrix:")
cm = confusion_matrix(true_labels, predictions)
print(cm)
print("[[TN FP]")
print(" [FN TP]]")

## Final Model Comparison
##
### Compare all three models side-by-side.

In [ ]:
print("\n" + "="*60)
print("FINAL MODEL COMPARISON: RoBERTa vs FinBERT vs Zero-Shot LLM")
print("="*60)

comparison_data = {
    "Model": [
        "RoBERTa (fine-tuned)",
        "FinBERT (fine-tuned)",
        f"Zero-Shot LLM ({MODEL_NAME.split('/')[-1]})"
    ],
    "Training": [
        "Fine-tuned on task",
        "Fine-tuned on task",
        "No training (zero-shot)"
    ],
    "Accuracy": [
        roberta_results['finetuned_accuracy'],
        finbert_results['finetuned_accuracy'],
        accuracy
    ],
    "Precision": [
        roberta_results.get('finetuned_precision', 0.95),
        finbert_results.get('finetuned_precision', 0.96),
        precision
    ],
    "Recall": [
        roberta_results.get('finetuned_recall', 0.95),
        finbert_results.get('finetuned_recall', 0.96),
        recall
    ],
    "F1-Score": [
        roberta_results.get('finetuned_f1', 0.95),
        finbert_results.get('finetuned_f1', 0.96),
        f1
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

# Determine winner
accuracies = comparison_df['Accuracy'].values
best_idx = int(np.argmax(accuracies))
winner = comparison_df.iloc[best_idx]['Model']
best_acc = accuracies[best_idx]

print(f"\n🏆 WINNER: {winner}")
print(f"   Best Accuracy: {best_acc:.4f}")

# Save comparison
final_comparison_path = os.path.join(RESULTS_DIR, "final_model_comparison.csv")
comparison_df.to_csv(final_comparison_path, index=False)
print(f"\nFinal comparison saved to: {final_comparison_path}")

## Analysis & Insights

In [ ]:
print("\n" + "="*60)
print("ANALYSIS & INSIGHTS")
print("="*60)

# Compare zero-shot vs fine-tuned
avg_finetuned = (roberta_results['finetuned_accuracy'] + finbert_results['finetuned_accuracy']) / 2
gap = avg_finetuned - accuracy

print(f"\nZero-Shot vs Fine-Tuned Performance:")
print(f"  Average fine-tuned accuracy: {avg_finetuned:.4f}")
print(f"  Zero-shot LLM accuracy:      {accuracy:.4f}")
print(f"  Performance gap:             {gap:.4f} ({gap*100:.2f}%)")

if gap > 0.1:
    print("\n💡 FINDING: Fine-tuning provides significant advantage")
    print("   → Task-specific training outperforms zero-shot prompting")
elif gap > 0:
    print("\n💡 FINDING: Fine-tuning provides modest advantage")
    print("   → Zero-shot LLM is competitive but slightly behind")
else:
    print("\n💡 FINDING: Zero-shot LLM matches or exceeds fine-tuned models")
    print("   → Large models can perform well without training data")

# Model efficiency comparison
print("\nModel Trade-offs:")
print("  RoBERTa:     Fast inference, requires fine-tuning, 82M parameters")
print("  FinBERT:     Fast inference, domain-specific, requires fine-tuning, 110M parameters")
print(f"  Zero-Shot:   Slower inference, no training needed, ~{MODEL_NAME.split('-')[-1] if 'B' in MODEL_NAME else '?'} parameters")

## Save Results

In [ ]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Save detailed results as JSON
results_json = {
    "model": MODEL_NAME,
    "backend": LLM_BACKEND,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "num_samples": int(len(eval_df)),
    "training": "zero-shot"
}

results_path = os.path.join(RESULTS_DIR, "zeroshot_llm_results.json")
with open(results_path, 'w') as f:
    json.dump(results_json, f, indent=2)

print(f"✓ Results saved to: {results_path}")